# ⚖️ 10 — Custom Judges

The built-in metrics in notebooks 04–09 answer general questions: is the answer faithful, did the
agent pick the right tool. Every store also has **house rules** no library ships a metric for:
how replies should sound, how policy must be worded, what a handover to a human must say. This
notebook writes judges for those rules, in three shapes, and saves every verdict so notebook 11
can check the judges against people.

## Learning Objectives
In this notebook, you will learn:
1. **Rubric judges** - `GEval` with your own evaluation steps and a pass/fail verdict, for tone, policy wording and escalation phrasing
2. **Scoping a judge** - each rule is judged only on the replies it applies to; "not applicable" is not a pass
3. **Decision-tree judges** - a `DAGMetric` that walks the refund flow one small question at a time
4. **Pairwise judges** - comparing two app versions answer by answer, with the positions swapped to catch position bias
5. **Judges next to free checks** - where a judge and an exact check disagree, and what each one misses

## Prerequisites
- Notebook 03 (`data/golden_set.json`) and notebook 04 (`data/golden_runs.json`, the baseline run of
  every golden case). `data/golden_runs_gpt-4o-mini.json`, the same 121 cases run on the first app model,
  ships with the repository
- `NEBIUS_API_KEY` (the judge runs on Nebius) in the `.env` file at the repository root. Nothing here
  runs the agent, so no other key is needed

---
## 🗺️ The idea

```
 saved runs (no agent calls)
    │
    ├── 1. three GEval rubrics ──────── tone (every reply), policy wording (RAG answers),
    │                                   escalation phrasing (replies that hand over to a human)
    ├── 2. a DAGMetric decision tree ── the refund flow: verified? → what happened? → said right?
    └── 3. a pairwise judge ─────────── current app vs the first app, each pair judged twice,
                                        once in each order
                          │
                          ▼
          every verdict and its reason → data/custom_judge_scores.json → notebook 11 (human labels)
```

Three rules hold for every judge here: the verdict is **binary or close to it** (people can label
pass/fail; they can't agree on 0.73), the judge **gives its reason before its verdict**, and the
judge is a **different model from the app** (Qwen judging DeepSeek), so it never grades its own work.

---
## ⚙️ Part 0: Setup

Every judge in this notebook reads saved runs, so the agent never runs and nothing is traced.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: keys, telemetry, judge model, the golden set and two recorded runs
# ============================================================================
import os
import json
from collections import Counter, defaultdict

from dotenv import load_dotenv

load_dotenv(os.path.join("..", "..", ".env"))
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"   # no DeepEval usage telemetry
os.environ["LANGSMITH_TRACING"] = "false"         # nothing to trace: the agent doesn't run here

from deepeval.models import GPTModel

# The judge: Qwen3 235B (Instruct 2507), on Nebius Token Factory. It is a different model family
# from the app (DeepSeek), so it never grades its own answers. Nebius speaks the OpenAI API:
# DeepEval takes a model object; RAGAS takes a client and the name.
JUDGE_MODEL_NAME = "Qwen/Qwen3-235B-A22B-Instruct-2507"   # an instruct model: no thinking to switch off
JUDGE_BASE_URL = "https://api.tokenfactory.nebius.com/v1"   # Nebius's OpenAI-compatible endpoint
JUDGE_KEY = os.getenv("NEBIUS_API_KEY") or "missing"   # never fall back to the OpenAI key
JUDGE_MODEL = GPTModel(model=JUDGE_MODEL_NAME, base_url=JUDGE_BASE_URL, api_key=JUDGE_KEY,
                       temperature=0)   # temperature 0: the same case gets the same verdict each run
JUDGE_MODEL.model_data.supports_structured_outputs = True   # Nebius enforces JSON schemas: let DeepEval ask for one

if not os.path.exists(os.path.join("data", "golden_runs.json")):
    raise RuntimeError("No data/golden_runs.json. Run notebook 04 first: it records the baseline run.")
with open(os.path.join("data", "golden_set.json"), encoding="utf-8") as f:
    golden_set = json.load(f)
with open(os.path.join("data", "golden_runs.json"), encoding="utf-8") as f:
    golden_runs = json.load(f)          # the current app: DeepSeek V4 Flash
with open(os.path.join("data", "golden_runs_gpt-4o-mini.json"), encoding="utf-8") as f:
    previous_runs = json.load(f)        # the first app version: gpt-4o-mini on the same 121 cases
cases_by_id = {case["id"]: case for case in golden_set}

print("✅ Setup complete; judge model:", JUDGE_MODEL_NAME)
print("🔑 NEBIUS_API_KEY set:", bool(os.getenv("NEBIUS_API_KEY")))
print(f"📂 {len(golden_set)} golden cases, {len(golden_runs)} current runs, {len(previous_runs)} previous runs")

In [ ]:
# ============================================================================
# WHAT THE AGENT DID, AS PLAIN FACTS A JUDGE CAN READ
# ============================================================================
# Written by code from the run record, never by a model. A judge that has to guess what happened
# from the reply alone would trust the reply; these facts let it check the reply against reality.
def actions_text(run):
    tools = [f"{call['name']}({', '.join(f'{k}={v}' for k, v in call['args'].items())})"
             for call in run["tool_calls"]] or ["none"]
    refunds = [f"{refund['order_id']} (${refund['amount']})" for refund in run["refunds_issued"]] or ["none"]
    tickets = [f"{ticket['ticket_id']} ({ticket['priority']})" for ticket in run["tickets_created"]] or ["none"]
    return (f"What the assistant did. Tools called: {'; '.join(tools)}. Refunds issued: {', '.join(refunds)}. "
            f"Tickets created for a human agent: {', '.join(tickets)}.")

example = next(run for run in golden_runs if run["refunds_issued"])
print("📋", actions_text(example))

---
## 📏 Part 1: Rubric judges with `GEval`

A **rubric judge** is a list of checks written in plain English. `GEval` can write its own steps
from a one-line `criteria`, but then the steps change whenever the wording changes, and nobody
reviewed them. Here the steps are written out (`evaluation_steps`), so they are the rubric: read,
argued over and versioned like code.

`strict_mode=True` makes each judge answer **pass (1) or fail (0)** instead of a score between 0
and 1. That costs nuance, and buys something notebook 11 needs: a person can label the same case
pass or fail, and the two can be compared.

### Key Concepts:
- **Evaluation steps**: the checks the judge must apply, in order; the last step says what passes
- **Evaluation params**: the only fields the judge sees. A judge can't use a field it isn't given,
  and it can't be distracted by one it isn't given either

In [ ]:
# ============================================================================
# THREE HOUSE RULES AS THREE RUBRICS
# ============================================================================
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams

tone_judge = GEval(
    name="Tone",
    evaluation_steps=[
        "Read the customer's message and the assistant's reply.",
        "The reply must be polite and calm, even when the customer is rude, upset or demanding.",
        "It must not blame the customer, lecture them, or sound sarcastic or cold.",
        "It must be plain and to the point: no filler, and at most one apology.",
        "Pass only if every point holds. A correct reply that is curt or long-winded fails.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=JUDGE_MODEL, strict_mode=True, async_mode=False,
)

policy_wording_judge = GEval(
    name="Policy wording",
    evaluation_steps=[
        "The retrieval context is the support articles the assistant was given.",
        "Find every store rule the reply states: a time window, a money limit, a fee, a condition.",
        "Each rule must use the articles' exact numbers and conditions ('30 days from delivery', not 'about a month').",
        "A rule the articles state firmly must not be softened ('usually', 'you may be able to'), and no "
        "exception the articles don't give may be added.",
        "Each stated rule must be followed by the chunk ID it came from, in square brackets.",
        "Pass only if every stated rule meets all three points. A reply that states no rule passes.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.RETRIEVAL_CONTEXT],
    model=JUDGE_MODEL, strict_mode=True, async_mode=False,
)

escalation_judge = GEval(
    name="Escalation phrasing",
    evaluation_steps=[
        "The context says what the assistant actually did, including any ticket created for a human agent.",
        "If a ticket was created, the reply must say that a human agent will follow up by email, within "
        "1 business day (4 hours for a safety incident).",
        "If no ticket was created, the reply must not claim that one was, or that a person will be in touch.",
        "The reply must not promise what the human will decide (for example 'your refund will be approved').",
        "It must not ask the customer to repeat details they already gave.",
        "Pass only if every point holds.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT],
    model=JUDGE_MODEL, strict_mode=True, async_mode=False,
)
print("✅ Three rubric judges ready:", [judge.name for judge in (tone_judge, policy_wording_judge, escalation_judge)])

### 1.1 🎯 Which replies each rule applies to

A rule judged on replies it doesn't apply to inflates the pass rate: a "how long is the return
window?" answer trivially passes escalation phrasing, because it never escalates. So each judge
gets only its own runs:
- **Tone**: every reply
- **Policy wording**: answers from the RAG agent to answerable questions (`simple`, `multi_hop`)
- **Escalation phrasing**: replies where a ticket was created or the supervisor sent the case to escalation

In [ ]:
# ============================================================================
# SCOPE: THE RUNS EACH RUBRIC JUDGES
# ============================================================================
tone_runs = golden_runs
policy_runs = [run for run in golden_runs if run["route"] == "rag_agent" and run["slice"] in ("simple", "multi_hop")]
escalation_runs = [run for run in golden_runs if run["tickets_created"] or run["route"] == "escalation"]

# Each judge gets a test case holding exactly the fields its evaluation_params name.
rubric_plan = [
    ("tone", tone_judge, tone_runs,
     lambda run: LLMTestCase(input=run["question"], actual_output=run["answer"])),
    ("policy_wording", policy_wording_judge, policy_runs,
     lambda run: LLMTestCase(input=run["question"], actual_output=run["answer"], retrieval_context=run["contexts"])),
    ("escalation_phrasing", escalation_judge, escalation_runs,
     lambda run: LLMTestCase(input=run["question"], actual_output=run["answer"], context=[actions_text(run)])),
]
for name, _, runs, _ in rubric_plan:
    print(f"🎯 {name:<20} {len(runs):>3} runs  {dict(Counter(run['slice'] for run in runs))}")
print(f"🧮 {sum(len(runs) for _, _, runs, _ in rubric_plan)} judge calls in total")

In [ ]:
# ============================================================================
# RUN THE THREE RUBRICS
# ============================================================================
# ⏱️ About 190 judge calls with the Qwen3 judge (one per run per rubric): well under a cent a call.
rubric_results = []
for name, judge, runs, make_test_case in rubric_plan:
    for run in runs:
        judge.measure(make_test_case(run), _show_indicator=False)   # no progress spinner per case
        rubric_results.append({"judge": name, "case_id": run["case_id"], "run_id": run["run_id"],
                               "slice": run["slice"], "passed": judge.score == 1, "reason": judge.reason})
    passed = sum(result["passed"] for result in rubric_results if result["judge"] == name)
    print(f"⚖️ {name:<20} {passed} of {len(runs)} passed")

In [ ]:
# ============================================================================
# PASS RATE PER RUBRIC AND SLICE
# ============================================================================
table = defaultdict(lambda: [0, 0])          # (judge, slice) → [passed, judged]
for result in rubric_results:
    cell = table[(result["judge"], result["slice"])]
    cell[0] += result["passed"]
    cell[1] += 1

slices = ["simple", "multi_hop", "out_of_scope", "tool_required", "handoff", "adversarial"]
print(f"{'judge':<20}" + "".join(f"{s:>15}" for s in slices))
for name, _, _, _ in rubric_plan:
    row = ""
    for s in slices:
        passed, judged = table[(name, s)]
        row += (f"{passed}/{judged}" if judged else "—").rjust(15)
    print(f"{name:<20}{row}")
print("Reading: '—' means the rule doesn't apply to that slice, which is different from 0 passes.")

In [ ]:
# ============================================================================
# READ THE FAILURES: is the judge right, or is the rubric wrong?
# ============================================================================
for name, _, _, _ in rubric_plan:
    failures = [result for result in rubric_results if result["judge"] == name and not result["passed"]]
    print(f"\n❌ {name}: {len(failures)} failures (first three)")
    for result in failures[:3]:
        run = next(run for run in golden_runs if run["run_id"] == result["run_id"])
        print(f"   {result['case_id']}")
        print(f"   ❓ {run['question'][:160]}")
        print(f"   🤖 {run['answer'][:220]}")
        print(f"   ⚖️ {result['reason'][:260]}")
print("\nReading: a failure is a claim. When the reason quotes a rule the rubric never stated, the")
print("rubric needs a clearer step, not the app a fix. Notebook 11 measures this with human labels.")

### 1.2 🆓 A judge beside a free check

Part of the escalation rule is exact: a reply that hands over must mention the response time.
A string search checks that for free. Where the two disagree, one of them is wrong, and reading
those cases is the quickest way to learn what each one misses.

In [ ]:
# ============================================================================
# ESCALATION: THE JUDGE vs "DOES THE REPLY MENTION THE RESPONSE TIME?"
# ============================================================================
runs_by_id = {run["run_id"]: run for run in golden_runs}
agreement = Counter()
for result in rubric_results:
    if result["judge"] != "escalation_phrasing":
        continue
    answer = runs_by_id[result["run_id"]]["answer"].lower()
    mentions_time = "business day" in answer or "4 hours" in answer
    agreement[(result["passed"], mentions_time)] += 1

print(f"{'':<16}{'time mentioned':>16}{'not mentioned':>15}")
for passed in (True, False):
    print(f"{'judge ' + ('pass' if passed else 'fail'):<16}{agreement[(passed, True)]:>16}{agreement[(passed, False)]:>15}")
print("Reading: 'pass + not mentioned' means the judge let a missing time slide (or no ticket was due);")
print("'fail + mentioned' means the reply had the time but broke another step, such as promising the outcome.")

---
## 🌳 Part 2: A decision-tree judge with `DAGMetric`

The refund flow has branches: was the customer verified? Then, was a refund issued, handed to a
human, or refused? What a good reply says depends on the branch. One `GEval` asked to "judge the
refund reply" must work out the branch and the verdict in one go, and when it gets it wrong
there is no telling which part failed.

A **`DAGMetric`** splits that into small questions, each answered by the judge on its own:

```
 customer gave order ID and email?        (reads the message)
   ├── no  → reply asks for them, promises nothing?             → 10 or 0
   └── yes → what happened?               (reads the facts written by code)
               ├── a refund was issued    → reply confirms it, with the amount?            → 10 or 0
               ├── a ticket was created   → reply explains the handover, claims no refund? → 10 or 0
               └── neither                → reply names the rule, claims no refund?        → 10 or 0
```

The scores sit on the leaves and are fixed by you, not by the judge, and the path the judge took
shows which question it answered how. `DAGMetric` divides the leaf score by 10.

> **Note**: the judge checks the **reply**, not the decision. A wrongly refused refund, explained
> politely with a real rule, passes this judge. The end state in the mock database is what catches
> a wrong decision; 2.2 puts the two side by side.

In [ ]:
# ============================================================================
# THE REFUND-FLOW DECISION TREE
# ============================================================================
from deepeval.metrics import DAGMetric
from deepeval.metrics.dag import DeepAcyclicGraph, BinaryJudgementNode, NonBinaryJudgementNode

def reply_check(criteria):
    # A leaf question about the reply: yes scores 10, no scores 0.
    node = BinaryJudgementNode(criteria=criteria,
                               evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.CONTEXT])
    node.add_verdict(True, score=10)
    node.add_verdict(False, score=0)
    return node

# Question 1, from the message alone: did the customer verify?
verified = BinaryJudgementNode(
    criteria="Does the customer's message give BOTH an order ID (like PC-10003) AND an email address?",
    evaluation_params=[SingleTurnParams.INPUT],
)
verified.add_verdict(False, then=reply_check(
    "The customer did not give both an order ID and an email. Does the reply ask for what is missing "
    "(or hand over to a human) WITHOUT issuing, confirming or promising a refund?"))

# Question 2, from the facts code wrote: what did the assistant actually do?
outcome = NonBinaryJudgementNode(
    criteria="According to the context, which of these happened?",
    evaluation_params=[SingleTurnParams.CONTEXT],
)
outcome.add_verdict("a refund was issued", then=reply_check(
    "A refund was issued. Does the reply confirm it, with the amount shown in the context, without "
    "promising anything else?"))
outcome.add_verdict("a ticket was created for a human agent", then=reply_check(
    "The request was handed to a human. Does the reply say why (for example over the $500 limit, or "
    "outside the return window), say that a human agent will reply by email, and NOT say a refund was issued?"))
outcome.add_verdict("no refund and no ticket", then=reply_check(
    "Nothing was refunded or handed over. Does the reply name the store rule that prevents the refund "
    "(sale window, return window, not delivered, already refunded, or details that don't match), and "
    "NOT say or imply that a refund was issued?"))
verified.add_verdict(True, then=outcome)

refund_judge = DAGMetric(name="Refund flow", dag=DeepAcyclicGraph(root_nodes=[verified]),
                         model=JUDGE_MODEL, async_mode=False)
print("✅ Refund-flow judge ready: 1 root question, 1 branch question, 4 leaf checks")

### 2.1 💵 Judge every refund conversation

The tree applies to messages that ask for a refund and go to the account side of the app: the
`tool_required`, `handoff` and `adversarial` cases that mention a refund. Each run costs two or
three tree questions plus one call to write the reason.

In [ ]:
# ============================================================================
# RUN THE TREE ON THE REFUND RUNS
# ============================================================================
# ⏱️ About 26 runs × 3–4 judge calls with the Qwen3 judge: well under a cent a call.
refund_runs = [run for run in golden_runs
               if run["slice"] in ("tool_required", "handoff", "adversarial")
               and "refund" in cases_by_id[run["case_id"]]["input"].lower()]

dag_results = []
for run in refund_runs:
    test_case = LLMTestCase(input=run["question"], actual_output=run["answer"], context=[actions_text(run)])
    refund_judge.measure(test_case, _show_indicator=False)
    dag_results.append({"case_id": run["case_id"], "run_id": run["run_id"], "slice": run["slice"],
                        "score": refund_judge.score, "passed": refund_judge.success, "reason": refund_judge.reason})

passed = sum(result["passed"] for result in dag_results)
print(f"🌳 {len(dag_results)} refund runs judged: {passed} passed, {len(dag_results) - passed} failed")
for s in ("tool_required", "handoff", "adversarial"):
    in_slice = [result for result in dag_results if result["slice"] == s]
    print(f"   {s:<14} {sum(r['passed'] for r in in_slice)}/{len(in_slice)}")

### 2.2 🔎 The reply judge beside the end state

Every refund case in the golden set records the **expected end state**: which orders should be
refunded and whether a ticket should exist. That is an exact check of the *decision*; the tree is a
judged check of the *reply*. Together they make four cells, and each disagreement cell has a name.

In [ ]:
# ============================================================================
# DAG VERDICT vs END STATE (exact, free)
# ============================================================================
def end_state_correct(run):
    expected = cases_by_id[run["case_id"]]["expected_end_state"]
    refunded = sorted(refund["order_id"] for refund in run["refunds_issued"])
    return refunded == sorted(expected["refund_order_ids"]) and bool(run["tickets_created"]) == expected["ticket_created"]

refund_runs_by_id = {run["run_id"]: run for run in refund_runs}
cells = defaultdict(list)
for result in dag_results:
    cells[(result["passed"], end_state_correct(refund_runs_by_id[result["run_id"]]))].append(result["case_id"])

print(f"{'':<14}{'end state right':>17}{'end state wrong':>17}")
for passed in (True, False):
    print(f"{'reply ' + ('pass' if passed else 'fail'):<14}{len(cells[(passed, True)]):>17}{len(cells[(passed, False)]):>17}")
print("\n🟡 Reply passed, decision wrong (only the end state catches these):", cells[(True, False)])
print("🟡 Decision right, reply failed (only the judge catches these):   ", cells[(False, True)])
print("Reading: neither check alone is enough. A release gate wants both cells empty.")

---
## 🥊 Part 3: A pairwise judge, with the positions swapped

Absolute scores drift: "is this answer a 1?" depends on the judge's mood that day. "Which of these
two answers is better?" is an easier question, and the one an app change actually raises. Here the
two versions are the **current app** (DeepSeek V4 Flash) and the **first app** (gpt-4o-mini), on the
same answerable questions.

Judges have a known habit: they favour the answer shown **first** (or, for some models, second).
So every pair is judged twice, once in each order. A winner counts only if it wins **both times**;
a pair whose winner changes with the order is a **position flip**, and it is evidence about the
judge, not about the answers.

> **Note**: DeepEval's `ArenaGEval` compares contestants too, but it shuffles their order at random
> on every call. That spreads position bias around instead of measuring it, so here the order is
> set by hand, with a small prompt and a typed answer.

In [ ]:
# ============================================================================
# THE PAIRWISE JUDGE: one prompt, a typed verdict, reason first
# ============================================================================
from typing import Literal
from pydantic import BaseModel

class PairVerdict(BaseModel):
    reason: str                          # first, so the judge explains before it decides
    winner: Literal["A", "B", "tie"]

PAIRWISE_PROMPT = """You compare two replies from a customer-support assistant to the same question.

Question: {question}

Reference answer (what a correct reply must say): {reference}

Reply A: {reply_a}

Reply B: {reply_b}

Which reply is better? A better reply agrees with the reference, gives the customer everything they
need, adds nothing the reference contradicts, and is shorter when both are equally correct.
Give a short reason, then the winner: "A", "B", or "tie" when neither is better."""

def compare(question, reference, reply_a, reply_b):
    prompt = PAIRWISE_PROMPT.format(question=question, reference=reference, reply_a=reply_a, reply_b=reply_b)
    verdict, _ = JUDGE_MODEL.generate(prompt, schema=PairVerdict)   # (parsed verdict, cost)
    return verdict

print("✅ Pairwise judge ready")

In [ ]:
# ============================================================================
# JUDGE EVERY PAIR TWICE: current first, then previous first
# ============================================================================
# ⏱️ Two judge calls per pair with the Qwen3 judge; identical replies are skipped (nothing to compare).
previous_by_case = {run["case_id"]: run for run in previous_runs}
answerable = [run for run in golden_runs if run["slice"] in ("simple", "multi_hop")]
pairs = [run for run in answerable if run["answer"].strip() != previous_by_case[run["case_id"]]["answer"].strip()]

pairwise_results = []
for run in pairs:
    case = cases_by_id[run["case_id"]]
    previous = previous_by_case[run["case_id"]]
    first = compare(case["input"], case["reference_answer"], run["answer"], previous["answer"])    # current is A
    second = compare(case["input"], case["reference_answer"], previous["answer"], run["answer"])   # current is B
    # Translate each verdict from a position (A or B) back to an app version.
    pick_first = {"A": "current", "B": "previous", "tie": "tie"}[first.winner]
    pick_second = {"A": "previous", "B": "current", "tie": "tie"}[second.winner]
    pairwise_results.append({
        "case_id": case["id"], "slice": case["slice"],
        "positions": [first.winner, second.winner],                # what the judge picked, by position
        "picks": [pick_first, pick_second],                        # the same picks, by app version
        "winner": pick_first if pick_first == pick_second else "position_flip",
        "reasons": [first.reason, second.reason],
    })
print(f"🥊 {len(pairs)} pairs judged twice ({2 * len(pairs)} calls); "
      f"{len(answerable) - len(pairs)} identical replies skipped")

In [ ]:
# ============================================================================
# THE RESULT: who won, and how much did the order matter?
# ============================================================================
winners = Counter(result["winner"] for result in pairwise_results)
decided = winners["current"] + winners["previous"]
print(f"🏆 current wins {winners['current']}, previous wins {winners['previous']}, "
      f"ties {winners['tie']}, position flips {winners['position_flip']}")
if decided:
    print(f"   current app's win rate on decided pairs: {winners['current'] / decided:.0%}")

# Position bias: of all verdicts that named a winner, how many named whichever reply came first?
named = [position for result in pairwise_results for position in result["positions"] if position != "tie"]
if named:
    print(f"📐 The reply shown first won {named.count('A')} of {len(named)} named verdicts "
          f"({named.count('A') / len(named):.0%}; an unbiased judge sits near 50%)")

for s in ("simple", "multi_hop"):
    in_slice = Counter(result["winner"] for result in pairwise_results if result["slice"] == s)
    print(f"   {s:<10} {dict(in_slice)}")
print("Reading: a flip is not a tie. It says the judge couldn't separate the replies, so the order decided.")

In [ ]:
# ============================================================================
# READ THE POSITION FLIPS
# ============================================================================
flips = [result for result in pairwise_results if result["winner"] == "position_flip"]
for result in flips[:3]:
    case = cases_by_id[result["case_id"]]
    print(f"🔄 {result['case_id']}: picked {result['picks'][0]} when current was first, {result['picks'][1]} when it was second")
    print(f"   ❓ {case['input'][:160]}")
    print(f"   ⚖️ 1st: {result['reasons'][0][:200]}")
    print(f"   ⚖️ 2nd: {result['reasons'][1][:200]}\n")
print(f"{len(flips)} flips in total. When flips are common, a pairwise result needs more cases, or a")
print("judge that gets the order right more often, before it can decide anything.")

---
## 💾 Part 4: Save every verdict for calibration

None of these judges is trusted yet. Notebook 11 samples their verdicts into a LangSmith
annotation queue, a person labels the same cases pass or fail, and agreement (Cohen's kappa) says
which judges can gate a release. The reasons are saved too: they are what a labeller reads to see
why a judge disagreed.

In [ ]:
# ============================================================================
# SAVE → data/custom_judge_scores.json
# ============================================================================
custom_judge_scores = {
    "judge_model": JUDGE_MODEL_NAME,
    "rubric": rubric_results,           # tone, policy_wording, escalation_phrasing: pass/fail + reason per run
    "refund_dag": dag_results,          # the refund-flow tree: leaf score, pass/fail + reason per run
    "pairwise": pairwise_results,       # current vs previous app: both orders, both reasons per case
}
with open(os.path.join("data", "custom_judge_scores.json"), "w", encoding="utf-8") as f:
    json.dump(custom_judge_scores, f, indent=2, ensure_ascii=False)
print(f"✅ Saved {len(rubric_results)} rubric, {len(dag_results)} tree and {len(pairwise_results)} pairwise "
      "verdicts → data/custom_judge_scores.json")

---
## 📝 Summary

In this notebook, we learned:

### 1. Rubric judges
- **Write the steps yourself**: `evaluation_steps` are the rubric, reviewed like code; `criteria`
  alone lets the judge invent them
- **Pass or fail**: `strict_mode=True` gives a verdict a person can also give, which makes
  calibration possible
- **Scope each rule**: a rule judged where it doesn't apply passes for free and hides failures

### 2. Decision-tree judges
- **Small questions**: `DAGMetric` asks one thing per node, and you fix the score on each leaf
- **Facts from code**: what the agent did is written by code, so the judge checks the reply against it
- **Reply vs decision**: the tree judges what was said; the end state judges what was done. A gate needs both

### 3. Pairwise judges
- **Compare, don't score**: "which is better?" is the question an app change raises
- **Swap the order**: a winner counts only if it wins in both positions; flips measure position bias

### Next Steps
- Notebook 11 puts a sample of these verdicts in a LangSmith annotation queue, collects human
  labels, and reports each judge's kappa, true-positive and true-negative rates, plus its
  position, verbosity and self-preference bias